# Lektion 11 - Modellkontextprotokoll (MCP)

Den **Modellkontextprotokoll (MCP)** är en öppen standard som gör det möjligt för agenter att dynamiskt upptäcka och använda verktyg, resurser och datakällor vid körning. Istället för att hårdkoda verktyg i en agent låter MCP agenter ansluta till externa servrar som exponerar funktioner på begäran.

I den här lektionen kommer du att lära dig:
- Vad MCP är och varför det är viktigt för agentsystem
- Hur MCP:s klient-serverarkitektur fungerar
- Hur man bygger agenter som använder MCP-liknande verktygsupptäckt


## Konfiguration

**Förutsättningar:**
- Azure AI Foundry-projekt med en driftsatt modell
- Kör `az login` för autentisering


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Vad är Model Context Protocol (MCP)?

MCP definierar ett standardiserat sätt för AI-agenter att upptäcka och interagera med externa verktyg och datakällor:

- **MCP Server**: Exponerar verktyg, resurser och prompts via ett standardprotokoll
- **MCP Client**: Agentens runtime som ansluter till servrar och upptäcker tillgängliga funktioner
- **Dynamic Discovery**: Agenter behöver inte hårdkodade verktyg — de upptäcker vad som är tillgängligt vid körning

Detta är kraftfullt för att bygga utbyggbara agentsystem där nya funktioner kan läggas till utan att ändra agentkoden.


## Hur MCP fungerar

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agenten (MCP-klient) ansluter till en MCP-server
2. Servern svarar med en lista över tillgängliga verktyg och deras scheman
3. Agenten kan sedan anropa vilket som helst av de upptäckta verktygen under sitt resonemang
4. Resultaten skickas tillbaka via samma protokoll


## Simulera MCP-verktygsupptäckt

Eftersom en riktig MCP-server kräver en körande serverprocess, kommer vi att demonstrera mönstret med hjälp av `@tool` funktioner som simulerar vad en MCP-ansluten boendetjänst skulle erbjuda.

I produktion skulle dessa verktyg upptäckas dynamiskt från en MCP-server istället för att definieras lokalt.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Bygga en agent med verktyg i MCP-stil


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP i produktion

I en produktionsmiljö möjliggör MCP kraftfulla mönster:

- **Dynamisk upptäckt av verktyg**: Agenter ansluter till MCP-servrar och upptäcker verktyg vid körning
- **Löst kopplad arkitektur**: Verktygsleverantörer kan uppdatera oberoende av agenten
- **Delning över organisationer**: Team kan exponera funktioner via MCP-servrar som vilken agent som helst kan använda
- **Microsoft Agent Framework-stöd**: MAF har inbyggt MCP-klientstöd via `mcp`-integrationen

För att använda en riktig MCP-server med MAF ansluter du via `hosted_mcp_tool()` eller MCP-klientintegrationen.

**Läs mer:**
- [MCP-specifikation](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP-stöd](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Sammanfattning

I den här lektionen lärde du dig:
- **MCP** är en öppen standard för dynamisk upptäckt av verktyg mellan agenter och verktygsleverantörer
- **Klient-serverarkitekturen** gör det möjligt för agenter att upptäcka funktioner vid körning
- MCP möjliggör **utbyggbara, löst kopplade agentsystem** där verktyg kan läggas till utan kodändringar
- Microsoft Agent Framework tillhandahåller **inbyggt MCP-stöd** för produktionsbruk


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Ansvarsfriskrivning:
Detta dokument har översatts med hjälp av AI-översättningstjänsten Co-op Translator (https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, vänligen observera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess originalspråk ska betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell, mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår genom användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
